# Stage 1 Evaluation

Six unsupervised evaluation tests for Sub-RQ 1.  
No ground truth required. No files written.

| Test | Name | What it checks |
|------|------|----------------|
| E-A | Bounds & sanity validation | Impossible values, extreme outliers, null distribution coherence |
| E-B | Mann-Whitney U | AQ Severity Z-Scores: Events With vs Without Citizen Reports |
| E-C | Rolling window sensitivity | Rank stability at 7 / 14 / 30 / 60-day windows (Saltelli et al. 2008) |


## Imports and Configuration

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

In [ ]:
# Relevant data
EVENTS_PATH = "./Processed/aggregated_events.parquet"
AQ_PATH     = "./Processed/aq_combined.parquet"

COKE_FINGERPRINT_CHEMS   = ["SO2", "H2S", "NO", "NOx"]
GENERAL_COMBUSTION_CHEMS = ["BC", "PM10", "PM2.5", "CO"]
TARGET_CHEMICALS         = COKE_FINGERPRINT_CHEMS + GENERAL_COMBUSTION_CHEMS + ["NO2"]

# Rolling window settings
ROLLING_DAYS  = 30          # pipeline default
MIN_PERIODS   = 72          # 72-hour minimum baseline
EVAL_WINDOWS  = [7, 14, 30, 60]  # sensitivity comparison windows (days)
EVAL_PERIOD   = "2025-03"   # evaluation month

# PG half-angles
PG_HALFANGLE = {"A": 45, "AB": 40, "B": 35, "BC": 30,
                "C": 25, "CD": 20, "D": 15, "E": 10, "F": 7}

# Z-score anomalous Thresholds
Z_EXTREME_THRESHOLD = 10.0


## Load Data

In [ ]:
ev = pd.read_parquet(EVENTS_PATH)
aq = pd.read_parquet(AQ_PATH)

dt_min = ev["event_datetime"].min().date()
dt_max = ev["event_datetime"].max().date()
n_monitors = aq["meetlocatie_id"].nunique()
print(f"Events : {len(ev)} rows | {dt_min} → {dt_max}")
print(f"AQ data: {len(aq)} rows | {n_monitors} monitors")


## Helper Functions

In [ ]:
def compute_rolling_zscore(df, window_days):
    # Identical replication of st1_aggregation.ipynb compute_rolling_zscore exactly.
    df = df.sort_values("begindatumtijd").reset_index(drop=True)
    window = window_days * 24
    for chem in TARGET_CHEMICALS:
        if chem not in df.columns:
            df[f"{chem}_z"] = np.nan
            continue
        roll_mean       = df[chem].rolling(window, min_periods=MIN_PERIODS).mean()
        roll_std        = df[chem].rolling(window, min_periods=MIN_PERIODS).std()
        df[f"{chem}_z"] = (df[chem] - roll_mean) / roll_std.replace(0, np.nan)
    return df


def severity_from_zscores(row):
    """Maximum sub-index operator — mirrors get_aq_at_peak in Stage_1.ipynb."""
    fp_vals  = [row.get(f"{c}_z", np.nan) for c in COKE_FINGERPRINT_CHEMS]
    gen_vals = [row.get(f"{c}_z", np.nan) for c in GENERAL_COMBUSTION_CHEMS]
    fp_z  = np.nanmax(fp_vals)  if any(np.isfinite(v) for v in fp_vals)  else np.nan
    gen_z = np.nanmax(gen_vals) if any(np.isfinite(v) for v in gen_vals) else np.nan
    vals  = [v for v in [fp_z, gen_z] if np.isfinite(v)]
    return max(vals) if vals else np.nan

## E-A  Bounds and Sanity Validation

**What it checks:** Impossible values, extreme outliers (|z| > 10), negative concentrations, out-of-range geometrical values, and whether null aq_severity_z values are structurally explained (offshore or no in-cone monitor).  

In [ ]:
# 1. Null value summary across all key columns
KEY_COLS = {
    "aq_severity_z":            "AQ severity (z)",
    "wind_speed_ms":            "Wind speed",
    "wind_dir_deg":             "Wind direction",
    "temp_c":                   "Temperature",
    "humidity_pct":             "Humidity",
    "radiation_jcm2":           "Solar radiation",
    "cloud_cover_n":            "Cloud cover",
    "plume_direction_deg":      "Plume direction",
    "n_postcodes_in_path":      "Postcodes in path",
    "primary_monitor":          "Primary monitor",
    "citizen_coke_specific":    "Citizen coke-specific reports",
    "concurrent_share":         "Concurrent share",
    "n_valid_citizen_reports":  "Valid citizen reports",
}
n_total = len(ev)
for col, label in KEY_COLS.items():
    if col not in ev.columns:
        print(f"  MISSING   {label} ({col})")
        continue
    n_null = ev[col].isna().sum()
    flag = "  !" if n_null > 0 else ""
    print(f"  {n_null:>4}/{n_total}  ({n_null / n_total * 100:>5.1f}%)  {label}{flag}")

# 2. Impossible / out-of-range meteorological & geometric values
BOUNDS = [
    ("wind_speed_ms",              0.0,   50.0,   "m/s",     ""),
    ("wind_dir_deg",               0.0,  360.0,   "°",       "360=0=N treated as edge; flag >360"),
    ("temp_c",                   -40.0,   60.0,   "°C",      ""),
    ("humidity_pct",               0.0,  100.0,   "%",       ""),
    ("radiation_jcm2",             0.0,   None,   "J/cm²",   "only negative checked; upper NaN-safe"),
    ("cloud_cover_n",              0.0,    8.0,   "oktas",   ""),
    ("plume_direction_deg",        0.0,  360.0,   "°",       ""),
    ("n_postcodes_in_path",        0.0,   None,   "count",   ""),
    ("concurrent_share",           0.0,    1.0,   "share",   ""),
    ("n_valid_citizen_reports",    0.0,   None,   "count",   ""),
    ("citizen_coke_specific",      0.0,   None,   "count",   ""),
]

n_bound_violations = 0
for col, lo, hi, unit, note in BOUNDS:
    if col not in ev.columns:
        print(f"  --        {col:<30}  not in dataset")
        continue
    s = ev[col].dropna()
    lo_viol = int((s < lo).sum()) if lo is not None else 0
    hi_viol = int((s > hi).sum()) if hi is not None else 0
    observed_range = f"[{s.min():.3f}, {s.max():.3f}]"
    expected_range = f"[{lo if lo is not None else '-∞'}, {hi if hi is not None else '+∞'}]"
    if lo_viol + hi_viol == 0:
        print(f"{col:<30}  expected {expected_range} {unit}  observed {observed_range}")
    else:
        n_bound_violations += lo_viol + hi_viol
        if lo_viol:
            print(f"!  {col:<30}  {lo_viol} values < {lo} {unit}  (min={s.min():.4f})")
        if hi_viol:
            print(f"!  {col:<30}  {hi_viol} values > {hi} {unit}  (max={s.max():.4f})")
        if note:
            print(f"       note: {note}")

# 3. Negative raw concentrations (aq_combined.parquet)
conc_cols = [c for c in TARGET_CHEMICALS if c in aq.columns]
if not conc_cols:
    print(" no TARGET_CHEMICALS columns found in aq")
n_neg_total = 0
for col in conc_cols:
    s = aq[col].dropna()
    n_neg = int((s < 0).sum())
    n_neg_total += n_neg
    if n_neg:
        print(f"!  {col:<8}  {n_neg}/{len(s)} negative  "
              f"({n_neg / len(s) * 100:.2f}%)  min={s.min():.4f}  max={s.max():.4f}")
    else:
        print(f"  {col:<8}  no negatives  "
              f"range=[{s.min():.4f}, {s.max():.4f}]  n={len(s)}")

# 4. Extreme z-score outliers (|z| > Z_EXTREME_THRESHOLD)
z_cols = ["aq_severity_z"] + sorted(c for c in ev.columns if c.startswith("z_"))
n_extreme_total = 0
for col in z_cols:
    if col not in ev.columns:
        continue
    s = ev[col].dropna()
    if s.empty:
        print(f" {col:<16}  all null — skipped")
        continue
    extreme_mask = s.abs() > Z_EXTREME_THRESHOLD
    n_ext = int(extreme_mask.sum())
    n_extreme_total += n_ext
    if n_ext:
        print(f"!  {col:<16}  {n_ext} events  max|z|={s.abs().max():.2f}")
        ext_idx = s[extreme_mask].index
        for _, r in ev.loc[ext_idx, ["event_datetime", "chimney", col]].iterrows():
            print(f"       {r['event_datetime']}  chimney={r['chimney']}  {col}={r[col]:.2f}")
    else:
        print(f"  {col:<16}  max|z|={s.abs().max():.2f}  "
              f"mean={s.mean():.3f}  std={s.std():.3f}  n={len(s)}")
        
# 5. Null distribution coherence (aq_severity_z)
null_z        = ev["aq_severity_z"].isna()
null_total    = null_z.sum()
null_offshore = (null_z & ev["offshore_flag"]).sum()
null_no_mon   = (null_z & ev["primary_monitor"].isna()).sum()
null_both     = (null_z & ev["offshore_flag"] & ev["primary_monitor"].isna()).sum()
null_union    = (null_z & (ev["offshore_flag"] | ev["primary_monitor"].isna())).sum()
null_unexpl   = null_total - null_union
pct_exp = null_union / null_total * 100 if null_total > 0 else 100

print(f"  aq_severity_z nulls: {null_total} total")
print(f"    Offshore only          : {null_offshore - null_both}")
print(f"    No monitor only        : {null_no_mon - null_both}")
print(f"    Both offshore+no mon   : {null_both}")
print(f"    Unexplained (neither)  : {null_unexpl}")

print("\n Summary")
print(f"  Bounds violations      : {n_bound_violations}")
print(f"  Negative concentrations: {n_neg_total}")
print(f"  Extreme z outliers     : {n_extreme_total}")
print(f"  Unexplained nulls      : {null_unexpl}/{null_total}  ({100 - pct_exp:.1f}% unexplained)")
overall = (n_bound_violations + n_neg_total + n_extreme_total + null_unexpl) == 0

## E-B Mann-Whitney U — AQ Severity Z-Scores: Events With vs Without Citizen Reports

In [ ]:
# Filter to events where AQ data is available
ev_aq = ev[ev["aq_severity_z"].notna()].copy()

has_reports = ev_aq[ev_aq["n_valid_citizen_reports"] > 0]["aq_severity_z"]
no_reports  = ev_aq[ev_aq["n_valid_citizen_reports"] == 0]["aq_severity_z"]

u_stat, p_val = stats.mannwhitneyu(has_reports, no_reports, alternative="greater")

direction = (
    "reversed (with < without)"
    if has_reports.mean() < no_reports.mean()
    else "expected (with > without)"
)

print(f"\nEvents with AQ data                   : {len(ev_aq)}")
print(f"  \u2014 with valid citizen reports          : {len(has_reports):>4d}  "
      f"mean AQ z = {has_reports.mean():.4f}")
print(f"  \u2014 without valid citizen reports       : {len(no_reports):>4d}  "
      f"mean AQ z = {no_reports.mean():.4f}")
print(f"Mann-Whitney U                        : {u_stat:.0f}")
print(f"p-value (one-sided, reports > none)   : {p_val:.4f}")
print(f"Direction                             : {direction}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(no_reports.values,  bins=30, alpha=0.65, color="steelblue",
        label="No citizen reports", density=True)
ax.hist(has_reports.values, bins=30, alpha=0.65, color="tomato",
        label="With citizen reports", density=True)
ax.set_xlabel("AQ severity z-score")
ax.set_ylabel("Density")
ax.set_title("E2 \u00b7 AQ severity z-score: events with vs without citizen reports")
ax.legend()
ax.text(0.98, 0.95, f"p = {p_val:.4f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=10)
plt.tight_layout()
plt.show()

## E-F Rolling Window Sensitivity

In [ ]:
march_ev = ev[ev["event_datetime"].dt.strftime("%Y-%m") == EVAL_PERIOD].copy()
march_ev = march_ev.dropna(subset=["primary_monitor", "event_datetime"])
print(f"March 2025 events with primary monitor: {len(march_ev)}")

# Normalise AQ timestamps to tz-naive
aq2 = aq.copy()
aq2["ts"] = aq2["begindatumtijd"].dt.tz_convert(None)

severity_by_window = {}

for window_days in EVAL_WINDOWS:
    print(f"  Computing {window_days}-day window...", end=" ")
    monitor_z = {}
    for mon_id, mon_df in aq2.groupby("meetlocatie_id"):
        cols = ["ts"] + [c for c in TARGET_CHEMICALS if c in mon_df.columns]
        work = mon_df[cols].sort_values("ts").reset_index(drop=True).copy()
        work = work.rename(columns={"ts": "begindatumtijd"})
        work = compute_rolling_zscore(work, window_days=window_days)
        work = work.rename(columns={"begindatumtijd": "ts"})
        monitor_z[mon_id] = work

    sev_list = []
    for _, row in march_ev.iterrows():
        mon_id  = row["primary_monitor"]
        ev_time = pd.Timestamp(row["event_datetime"])
        if mon_id not in monitor_z:
            sev_list.append(np.nan)
            continue
        mdf   = monitor_z[mon_id]
        diffs = (mdf["ts"] - ev_time).abs()
        nearest = diffs.idxmin()
        if diffs[nearest] > pd.Timedelta("90min"):
            sev_list.append(np.nan)
            continue
        sev_list.append(severity_from_zscores(mdf.iloc[nearest]))

    severity_by_window[window_days] = sev_list
    n_valid = sum(1 for v in sev_list if not np.isnan(v))
    print(f"{n_valid}/{len(march_ev)} valid")

sev_df = pd.DataFrame(severity_by_window, index=march_ev.index)
sev_df.columns = [f"{w}d" for w in EVAL_WINDOWS]

# Rank correlations vs 30-day baseline
print(f"Spearman rank correlations vs 30d baseline:")
print(f"  {'Window':<8} {'ρ':>8} {'p-value':>10} {'n':>6} {'Stability'}")
print(f"  {'-'*50}")
stable_count = 0
for col in ["7d", "14d", "60d"]:
    pair = sev_df[[col, "30d"]].dropna()
    if len(pair) < 5:
        continue
    rho, p = stats.spearmanr(pair[col], pair["30d"])
    flag = "STABLE" if rho > 0.85 else "SENSITIVE"
    if rho > 0.85:
        stable_count += 1
    print(f"  {col:<8} {rho:>8.4f} {p:>10.4f} {len(pair):>6}  {flag}")

print(f"Distribution per window (March 2025 events):")
print(f"  {'Window':<8} {'mean':>8} {'std':>8} {'max':>8} {'n>1.0':>8}")
print(f"  {'-'*42}")
for col in sev_df.columns:
    vals = sev_df[col].dropna()
    print(f"  {col:<8} {vals.mean():>8.3f} {vals.std():>8.3f} {vals.max():>8.3f} {(vals>1.0).sum():>8}")
